# Real Estate Due-Diligence Copilot — Capstone Notebook

OCR to RAG to 3-agent LangGraph system to FastAPI to AWS deployment.

Walkthrough for one property using the sample docs in `sample_docs/`.

## 0. Setup

In [ ]:
!pip install -q langchain langchain-community langchain-openai langgraph faiss-cpu pymupdf pytesseract pillow python-dotenv

import os
from dotenv import load_dotenv
load_dotenv()
assert os.getenv('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in your .env file'

## 1. OCR — extract text from property documents

PyMuPDF handles text-native PDFs directly (fast). Scanned pages fall back to
Tesseract OCR. This matters for real estate docs specifically because
encumbrance certificates and old sale deeds from sub-registrar offices are
very often scanned images, not text PDFs.

In [ ]:
from pipeline import load_documents

doc_texts = load_documents('sample_docs/')
for fname, text in doc_texts.items():
    print(f'--- {fname} ({len(text)} chars) ---')
    print(text[:300], '...\n')

## 2. RAG index — chunk, embed, store in FAISS

Chunked to 800 chars with 120 overlap — small enough for precise retrieval of
specific clauses (survey numbers, encumbrance entries) without losing context.

In [ ]:
from pipeline import build_vectorstore

vs = build_vectorstore(doc_texts)
results = vs.similarity_search('encumbrance mortgage pending charge', k=3)
for r in results:
    print(r.metadata, '->', r.page_content[:150])

## 3. Multi-agent system — 3 collaborating agents

1. **Document Verification Agent** — checks presence/completeness against a
   standard Indian property due-diligence checklist (sale deed, EC, tax
   receipts, khata, DTCP/HMDA approval, OC, link docs, survey sketch).
2. **Legal & Title Agent** — RAG-grounded, checks title chain, encumbrances,
   litigation, zoning against retrieved document excerpts.
3. **Risk Scoring Agent** — synthesizes both into a single buyer-facing
   verdict: LOW/MEDIUM/HIGH risk, ranked red flags, next actions.

In [ ]:
from pipeline import REQUIRED_DOCS
print('\n'.join(REQUIRED_DOCS))

## 4. Orchestration — LangGraph StateGraph wiring the 3 agents

In [ ]:
from pipeline import run_due_diligence

result = run_due_diligence('sample_docs/', 'Plot 14, Balanagar, Hyderabad')

print('=== DOCUMENT CHECKLIST ===')
print(result['checklist_report'])
print('\n=== LEGAL & TITLE FINDINGS ===')
print(result['legal_findings'])
print('\n=== FINAL RISK REPORT ===')
print(result['risk_report'])

## 5. FastAPI — serving the pipeline

See `app.py`. Three endpoints: `POST /upload`, `POST /analyze/{job_id}`,
`GET /report/{job_id}`, plus `GET /health` for load-balancer checks.

Run locally: `uvicorn app:app --reload` then hit the endpoints (see README
for curl examples).

## 6. AWS deployment

Dockerized (`Dockerfile`) and deployable three ways — ECS Fargate (recommended),
EC2 + Docker, or Lambda via Mangum for low-volume use. Full step-by-step
commands are in `README.md`. Key production swaps flagged there: in-memory
job store -> DynamoDB, local FAISS -> S3-backed index, add auth before
exposing endpoints publicly.